In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> TCGA-PAAD Tumor
>>> case Tumor
	DEGs 20484
		Up (#12140)
		Dw (#8344)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    13
1                      IG_C_pseudogene     3
2                            IG_J_gene     9
3                      IG_J_pseudogene     1
4                            IG_V_gene   127
5                      IG_V_pseudogene    48
6                              Mt_tRNA    17
7                                  TEC   148
8                            TR_C_gene     6
9                            TR_D_gene     1
10                           TR_J_gene     8
11                           TR_V_gene    45
12                     TR_V_pseudogene     4
13                              lncRNA  4162
14                               miRNA   139
15                            misc_RNA    36
16              polymorphic_pseudogene     4


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'


verbose=True
cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)


-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"


### Calc expression

 - calc_file_expression_tumor_normal_gtex()
   - get_dic_expression_tumor_and_normal()
     - get_filtered_tables()
     - get_table_given_fileID()
   - prepare_normal_tumor_tables()

  
#### Tables in: root_disease / lfc

In [8]:
cbio.root_disease, cbio.root_lfc, cbio.filename_demo

(PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/clinical_and_demographics_for_paad_cptac_2021.tsv'))

### Get cases, subtypes and clin_demo tables

In [ ]:
verbose=False
force=False

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'TCGA'
psi_id = 'SKCM'
psi_id = 'BRCA'
psi_id = 'PAAD'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, force=force, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape

((185, 27), (184, 43), (184, 2))

In [10]:
verbose=False

# dic_tumor, dic_normal = cbio.get_dic_expression_tumor_and_normal(verbose=verbose)

In [15]:
verbose=False
force=False

imax_tumor=200
imax_normal=100

df_tumor, df_normal, df_gtex_ctrl = cbio.calc_file_expression_tumor_normal_gtex(
            imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)

print(df_tumor.shape[1], df_normal.shape[1], df_gtex_ctrl.shape[1])

85 5 0


In [16]:
df_tumor.head(3)

,geneid,symbol,biotype,T-TCGA-IB-A6UG,T-TCGA-HZ-8637,T-TCGA-IB-8126,T-TCGA-HV-A5A6,T-TCGA-FB-A5VM,T-TCGA-3A-A9IB,T-TCGA-US-A77J,...,T-TCGA-HZ-7289,T-TCGA-3A-A9IN,T-TCGA-3A-A9IS,T-TCGA-2L-AAQM,T-TCGA-3A-A9IR,T-TCGA-3A-A9IV,T-TCGA-3A-A9IO,T-TCGA-2J-AABT,T-TCGA-H6-A45N,T-TCGA-3A-A9IJ
0,ENSG00000000003,TSPAN6,protein_coding,2684,5505,1734,3081,2139,2856,1044,...,2506,442,38,299,77,158,394,659,1294,395
1,ENSG00000000005,TNMD,protein_coding,0,49,47,4,91,2,1,...,2,172,4,2,1,2,14,3,3,0
2,ENSG00000000419,DPM1,protein_coding,1139,3152,1263,1515,2767,1507,1032,...,1638,940,1372,1008,1493,1059,922,839,717,1034


In [17]:
df_normal.head(3)

,geneid,symbol,biotype,N-TCGA-H6-8124,N-TCGA-H6-A45N
0,ENSG00000000003,TSPAN6,protein_coding,3738,369
1,ENSG00000000005,TNMD,protein_coding,4,5
2,ENSG00000000419,DPM1,protein_coding,1532,1023


In [ ]:
df_gtex_ctrl.head(3)

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

In [ ]:
cbio.df_metadata

In [ ]:
print(pd.crosstab(cbio.df_metadata["program"], cbio.df_metadata["condition"]))

### Batch effect correction and cpm normalization

In [ ]:
force=False
verbose=False

perc_min_samples=0.25; top_n=10_000

df_sel, df_cpm, df_gene_annot = cbio.calc_expression_and_batch(dfn_tumor=dfn_tumor, group='Tumor', 
                                                           perc_min_samples=perc_min_samples, top_n=top_n,
                                                           force=force, verbose=verbose)

dfn, df_gene_annot = cbio.calc_cpm_merge_turmor_and_normal(dfn_tumor=dfn_tumor, dfn_normal=dfn_normal, 
                                                                   perc_min_samples=perc_min_samples, top_n=top_n,
                                                                   verbose=verbose)

dfall = cbio.dfall
print(dfn.shape, dfall.shape)
dfall.tail(3)

In [ ]:
dfall.head(3).T

In [ ]:
dfn.head(3)

In [ ]:
dfn.shape, np.sum([1 if x.startswith('T-') else 0 for x in dfn.columns]), np.sum([1 if x.startswith('N-') else 0 for x in dfn.columns])

In [ ]:
dfall.shape

### Batch effect correction

In [ ]:
dfn_tumor.shape, dfn_normal.shape

In [ ]:
df_metadata = cbio.df_metadata
df_metadata.head(3)

### Combat

In [ ]:
verbose=False
force=False

df_combat = cbio.calc_cpm_merge_turmor_and_normal_batch_correction(dfn_tumor=dfn_tumor, dfn_normal=dfn_normal, 
                                                                   perc_min_samples=perc_min_samples, top_n=top_n,
                                                                   force=force, verbose=verbose)

print(df_combat.shape)
df_combat.head(3)

In [ ]:
cbio.plot_boxplot_combat(df_combat=df_combat, title="ComBat Corrected Expression across samples", figsize=(16, 6))

### Loop all clusters

In [ ]:
imax_tumor=250
imax_normal=50
exclude_prog_list=['CCLE']
n_components = 10
n_umap_neighbors=5; min_umap_dist=0.2; umap_metric="euclidean"
method_hca="ward"; hca_criterion="maxclust"
LFC_cutoff=1; FDR_cutoff=0.05

force=False
verbose=False

group = 'Tumor'
min_clusters = 3

run_all_custers=True

if run_all_custers:

    for n_clusters in range(3, 10+1):

        print(f"Clusters: {n_clusters}")

        fname = f'all_cluster_degs_for_{n_clusters}_clusterization.txt'
        filename = cbio.root_mprog_lfc / fname

        '''
        if filename.exists() and not force:
            print(f"\tFile {fname} already exists. Skipping.")
            continue
        '''
        # max_clusters is for silhouette
        df_cluster, df_pca, df_umap = cbio.cluster_PCA_HCA_UMAP(df_combat, group=group, n_clusters=n_clusters, 
                                                        n_components=n_components, min_clusters=min_clusters, max_clusters=n_clusters+2,
                                                        n_umap_neighbors=n_umap_neighbors, min_umap_dist=min_umap_dist, umap_metric=umap_metric,
                                                        method_hca=method_hca, hca_criterion=hca_criterion,
                                                        lfc_cutoff=LFC_cutoff, fdr_cutoff=FDR_cutoff,
                                                        force=force, verbose=verbose)
        
        all_clusters_text = ''

        for nclu in range(1, n_clusters + 1):

            print(f"\tProcessing cluster {nclu}/{n_clusters}")

            df_lfc, df_lfc_ori, degs_txt, degs_first2000, degs_for_AI_analysis, msg =  \
            cbio.calc_limma_inmoose(df_combat=df_combat, nclu=nclu, max_cluster=n_clusters, disease_cd=disease_cd, 
                                    age_cutoff=60, bmi_cutoff=32,
                                    lfc_cutoff=LFC_cutoff, fdr_cutoff=FDR_cutoff,
                                    force=force, verbose=verbose )
            
            text = f"\tCluster {nclu}:\n{degs_for_AI_analysis}"

            if all_clusters_text == '':
                all_clusters_text = text
            else:
                all_clusters_text += "\n\n----------------------------------\n" + text
            
            if 'has no valid sample' in msg:
                pass
            else:
                print(msg)
                print(f"\tHas {len(df_lfc)} DEGs")
            print('\n----------------------------------')


        print(f"------------------- end clusterization {n_clusters}---------------\n\n")
        write_txt(all_clusters_text, fname, cbio.root_mprog_lfc)

    print(f"\n------------------- end -----------------------")

In [ ]:
imax_tumor=250
imax_normal=50
exclude_prog_list=['CCLE']
n_components = 10
n_umap_neighbors=5; min_umap_dist=0.2; umap_metric="euclidean"
method_hca="ward"; hca_criterion="maxclust"
LFC_cutoff=1; FDR_cutoff=0.05

force=False
verbose=False

group = 'Tumor'
min_clusters = 3

n_clusters = 10

print(f"Clusters: {n_clusters}")

fname = f'all_cluster_degs_for_{n_clusters}_clusterization.txt'
filename = cbio.root_mprog_lfc / fname

# for silhouette
max_clusters = n_clusters+2

df_cluster, df_pca, df_umap = cbio.cluster_PCA_HCA_UMAP(df_combat, group=group, n_clusters=n_clusters, 
                                                n_components=n_components, min_clusters=min_clusters, max_clusters=n_clusters+2,
                                                n_umap_neighbors=n_umap_neighbors, min_umap_dist=min_umap_dist, umap_metric=umap_metric,
                                                method_hca=method_hca, hca_criterion=hca_criterion,
                                                lfc_cutoff=LFC_cutoff, fdr_cutoff=FDR_cutoff,
                                                force=force, verbose=verbose)

df_hca = cbio.df_hca
df_umap =cbio.df_umap
dfall = cbio.dfall

In [ ]:
n_clusters, cbio.primary_site, group

In [ ]:
dfall.tail(3)

In [ ]:
cbio.df_umap.tail(3)

In [ ]:
df_hca.tail(3)

In [ ]:
df_hca.cluster.unique()

### Normal clusters

In [ ]:
normal_samples = df_hca[df_hca['sample'].str.startswith('N-')]
normal_samples

In [ ]:
cbio.plot_PCA(df_pca, figsize= (10, 8))

In [ ]:
df_eval, df_samp_clusters = cbio.calc_best_cluster(df_pca, min_clusters=3, max_clusters=8)
df_eval

In [ ]:
df_samp_clusters.head(6)

### PCA-UMAP

In [ ]:
df_umap2 = df_umap.set_index('sample')
df_umap2

In [ ]:
n_neighbors=3
min_dist=0.2

cbio.plot_PCA_UMAP(df_umap2, n_neighbors=n_neighbors, min_dist=min_dist, figsize=(10,8))

In [ ]:
df_cluster

In [ ]:
df_umap

In [ ]:
np.unique(df_hca.cluster)


In [ ]:
df_hca.head(3)

In [ ]:
import plotly.graph_objects as go

colors = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "yellow",
    "cyan",
    "magenta",
    "black",
    "gray",
]

dic_groups = {
    1: "Control",
    2: "Control",
    3: "Pancreatic-lineage-low, acinar-depleted, lncRNA-high",
    4: "Basal-leaning, acinar-depleted",
    5: "Stromal-remodelling, SPP1-myeloid, basal-leaning",
    6: "Neuroendocrine-like, acinar-depleted",
    7: "Mucinous/GI-like classical-secretory",
    8: "Control",
    9: "Immune/plasma-cell and stromal-rich",
    10: "Strong basal-like/squamous, classical–basal hybrid",
}



fig = cbio.plot_PCA_UMAP_plotly(df_umap=df_umap, df_hca=df_hca, dic_groups=dic_groups, n_neighbors=n_neighbors, min_dist=min_dist, width=1300, height=1000)

if fig: fig.show()


### Hierarchical clustering alternative

For 32 samples, this is often better than UMAP.

In [ ]:
# Cut tree into n clusters
n_clusters=10

df_cluster_hca = cbio.cut_HCA_PCA(df_pca=df_pca, n_clusters=n_clusters, method="ward", criterion="maxclust", verbose=True)
df_cluster_hca.head(6)

In [ ]:
cbio.plot_HCA_PCA_UMAP(df_umap, figsize=(10, 8))

In [ ]:
# Cut tree into k clusters

df_cluster_umap = cbio.cut_HCA_PCA_UMAP(df_umap=df_umap, n_clusters=n_clusters, method="ward", criterion="maxclust", verbose=True)
df_cluster_umap.head(6)

### Define cluster marker genes

This finds genes high in one cluster compared with all others.

### cluster signatures

In [ ]:
df_cluster

In [ ]:
df_cluster_umap.cluster.unique()

In [ ]:
df_cluster_umap

### Development & tests

### Remove olders from control-normal

In [ ]:
age_cutoff = 60
bmi_cutoff = 32

df_psi = cbio.open_primary_site(verbose=False)

df_psi = df_psi[ (df_psi.disease_cd == disease_cd) & (~pd.isnull(df_psi.primary_site)) & (~pd.isnull(df_psi.cbioportal_study_id)) ]
dfa = df_psi.groupby(['prog_id', 'psi_id', 'disease_id', 'disease_cd', 'primary_site', 'cbioportal_study_id']).size().reset_index()


bad_list = []

for i, row in dfa.iterrows():

    _ = cbio.set_program_and_primary_site(prog_id=row.prog_id, psi_id=row.psi_id)

    fname_demo = cbio.fname_demo0 % row.cbioportal_study_id
    filename_demo = cbio.root_disease / cbio.fname_demo

    print(i, row.psi_id, cbio.psi_id, row.cbioportal_study_id, "-->", filename_demo)


    if not filename_demo.exists():
        print(f"Could not find demo for {filename_demo}")
        continue

    df_demo = pdreadcsv(fname_demo, cbio.root_disease, verbose=True)


    df_bad = df_demo[ (df_demo.age >= age_cutoff) | (df_demo.bmi >= bmi_cutoff) ]

    df_bad = ['N-'+x for x in df_bad.barcode_case]

    bad_list += df_bad


print(len(bad_list))
bad_list


In [ ]:
bad_cols = [x for x in df_combat.columns if x in bad_list]
print(len(bad_cols))
bad_cols


In [ ]:
cols = [x for x in df_combat.columns if x not in bad_cols]

have_tumor = np.sum([1 for x in cols if x.startswith('T-')])
have_normal = np.sum([1 for x in cols if x.startswith('N-')])

print(have_tumor, have_normal)

if have_normal >= 3:
    df_combat2 = df_combat[cols]
else:
    df_combat2 = df_combat

print(df_combat2.shape)
df_combat2

In [ ]:

def plot_PCA_UMAP_plotly(df_umap: pd.DataFrame, df_hca: pd.DataFrame, dic_groups: dict, n_neighbors: int, min_dist: float, width:int=800, height:int=600) -> go.Figure:



    fig = go.Figure()

    maxi = np.max(df_hca.cluster)

    icolor = -1
    for nclu in np.arange(1, maxi+1):

        if nclu <= 2:
            # control
            icolor = 0
            text = f"1: Control"
        elif nclu <= 4:
            # control
            icolor = 1
            text = f"{3}: {groups.get(3, 'Unknown')}"
        else:
            icolor += 1
            text = f"{nclu}: {groups.get(nclu, 'Unknown')}"

        color = colors[icolor]
        samp_list = df_hca[df_hca["cluster"] == nclu]['sample']
        
        df2 = df_umap[df_umap["sample"].isin(samp_list)]

        fig.add_trace(
            go.Scatter(
                x=df2["UMAP1"],
                y=df2["UMAP2"],
                mode="markers+text",
                name=text,
                text="",
                textposition="middle right",
                textfont=dict(size=11),
                marker=dict(
                    size=10,
                    color=color,
                ),
                customdata=df2["sample"],
                hovertemplate=(
                    "<b>%{customdata}</b><br>"
                    f"{text}<br>"
                    "UMAP1: %{x:.3f}<br>"
                    "UMAP2: %{y:.3f}"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=f"UMAP of {group} samples "
            f"(n_neighbors={n_neighbors}, min_dist={min_dist})",
        xaxis_title="UMAP1",
        yaxis_title="UMAP2",
        width=width,
        height=height,
        template="plotly_white",
        hovermode="closest",
    )

    return fig

